# Evaluation Results

Collected LLM-judge evaluation results across experiments.  
Each section below is one experiment with winrate and/or rubric tables.

**Sharing:** Run all cells, then share via:
- Push to GitHub → opens in nbviewer automatically
- `jupyter nbconvert --to html eval_results.ipynb` → share the HTML file
- Upload to [Google Colab](https://colab.research.google.com/) or [nbviewer](https://nbviewer.org/)

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.float_format", "{:.1f}".format)

RESULTS_ROOT = Path(
    "/work/dlclarge2/ferreira-oellm/open-instruct/oellm/evaluations/benchmarks/OpenJury/results"
)


def _short_model(name: str) -> str:
    """Extract a readable model name from the VLLM path."""
    path_str = name.split("VLLM/", 1)[-1] if "VLLM/" in name else name
    p = Path(path_str)
    if p.exists():
        try:
            p = p.resolve()
        except Exception:
            pass
    return p.name


def load_winrate(results_dir: str) -> pd.DataFrame:
    """Load winrate results from a results directory into a DataFrame."""
    base = RESULTS_ROOT / results_dir
    rows = []
    for f in sorted(base.rglob("results-*.json")):
        d = json.loads(f.read_text())
        if d.get("eval_mode") != "winrate":
            continue
        model_b_wr = 1 - d["winrate"]  # flip: winrate field is model_A's
        rows.append({
            "Benchmark": d["dataset"],
            "Baseline WR%": d["winrate"] * 100,
            "Ours WR%": model_b_wr * 100,
            "Battles": d["num_battles"],
            "Wins": d["num_losses"],   # model_B wins = model_A losses
            "Losses": d["num_wins"],   # model_B losses = model_A wins
            "Ties": d["num_ties"],
        })
    if not rows:
        print(f"No winrate results found in {base}")
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    # Add average row
    avg = df[["Baseline WR%", "Ours WR%"]].mean()
    avg_row = {"Benchmark": "**Average**", **avg.to_dict(), "Battles": df["Battles"].sum(),
               "Wins": df["Wins"].sum(), "Losses": df["Losses"].sum(), "Ties": df["Ties"].sum()}
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    return df


def load_rubric(results_dir: str) -> pd.DataFrame:
    """Load rubric results from a results directory into a DataFrame."""
    base = RESULTS_ROOT / results_dir
    rows = []
    for f in sorted(base.rglob("results-*.json")):
        d = json.loads(f.read_text())
        if d.get("eval_mode") != "rubric":
            continue
        a_scores = d["model_A_scores"]
        b_scores = d["model_B_scores"]
        for criterion in d["criteria"]:
            key = f"{criterion}_score"
            rows.append({
                "Benchmark": d["dataset"],
                "Criterion": criterion.replace("_", " ").title(),
                "Baseline": a_scores[key],
                "Ours": b_scores[key],
                "Delta": b_scores[key] - a_scores[key],
            })
        # Composite row
        rows.append({
            "Benchmark": d["dataset"],
            "Criterion": "**Composite**",
            "Baseline": a_scores["composite_score"] * 100,
            "Ours": b_scores["composite_score"] * 100,
            "Delta": (b_scores["composite_score"] - a_scores["composite_score"]) * 100,
        })
    if not rows:
        print(f"No rubric results found in {base}")
        return pd.DataFrame()
    return pd.DataFrame(rows)

---
## Think SFT v2 (HoreKa) — Winrate

| | |
|---|---|
| **Ours** | `dolci-think-sft-v2-horeka-hf` (`checkpoints/ferreira/olmo3-7b-sft/dolci-think-sft-v2-horeka-hf`) |
| **Baseline** | `Olmo-3-7B-Think-SFT` (`models/baselines/Olmo-3-7B-Think-SFT`) |
| **Judge** | `Qwen/Qwen3-30B-A3B-Instruct-2507` (winrate mode, both orderings) |
| **Date** | 2026-02-24 |

In [2]:
df_think_wr = load_winrate("horeka-winrate-Olmo-3-7B-Think-SFT-20260224_141545")
display(df_think_wr.style.format({
    "Baseline WR%": "{:.1f}",
    "Ours WR%": "{:.1f}",
    "Battles": "{:.0f}",
    "Wins": "{:.0f}",
    "Losses": "{:.0f}",
    "Ties": "{:.0f}",
}).hide(axis="index"))

Benchmark,Baseline WR%,Ours WR%,Battles,Wins,Losses,Ties
alpaca-eval,48.9,51.1,1610,819,783,8
arena-hard,49.5,50.5,1000,503,493,4
m-arena-hard-EU,48.8,51.2,12000,6129,5838,33
**Average**,49.1,50.9,14610,7451,7114,45


---
## Think SFT v2 (HoreKa) — Rubric

Per-criterion scores (1-10 scale, higher is better). Composite is a normalized 0-100 score.

| | |
|---|---|
| **Ours** | `dolci-think-sft-v2-horeka-hf` (`checkpoints/ferreira/olmo3-7b-sft/dolci-think-sft-v2-horeka-hf`) |
| **Baseline** | `Olmo-3-7B-Think-SFT` (`models/baselines/Olmo-3-7B-Think-SFT`) |
| **Judge** | `Qwen/Qwen3-30B-A3B-Instruct-2507` (rubric mode) |
| **Date** | 2026-02-24 |

In [3]:
df_think_rubric = load_rubric("horeka-rubric-Olmo-3-7B-Think-SFT-20260224_161039")
if not df_think_rubric.empty:
    _color_delta = lambda v: "color: green" if isinstance(v, (int, float)) and v > 0 else (
        "color: red" if isinstance(v, (int, float)) and v < 0 else "")
    _styler = df_think_rubric.style.format({
        "Baseline": "{:.2f}",
        "Ours": "{:.2f}",
        "Delta": "{:+.2f}",
    }).hide(axis="index")
    # pandas >=2.1 uses .map(), older uses .applymap()
    _map_fn = getattr(_styler, "map", getattr(_styler, "applymap", None))
    _map_fn(_color_delta, subset=["Delta"])
    display(_styler)
else:
    print("Rubric evaluation not yet available.")

Benchmark,Criterion,Baseline,Ours,Delta
alpaca-eval,Instruction Following,6.36,6.41,+0.05
alpaca-eval,Naturalness,6.81,6.77,-0.05
alpaca-eval,Coherence,6.70,6.66,-0.04
alpaca-eval,Accuracy,6.48,6.47,-0.01
alpaca-eval,**Composite**,93.13,92.95,-0.17
arena-hard,Instruction Following,5.80,5.82,+0.03
arena-hard,Naturalness,6.52,6.50,-0.02
arena-hard,Coherence,6.27,6.31,+0.03
arena-hard,Accuracy,5.89,5.95,+0.06
arena-hard,**Composite**,85.31,85.72,+0.41


---
## Instruct SFT (placeholder)

Add instruct experiment results here once available. Copy the pattern above:
```python
df_instruct_wr = load_winrate("<instruct-winrate-dir>")
df_instruct_wr.style.format({...}).hide(axis="index")
```